<a href="https://colab.research.google.com/github/danielcosta04/LinguagemProgramacao/blob/main/C%C3%B3pia_de_Atividade_Pr%C3%A1tica_%E2%80%94_Aula_2_Pandas_Essencial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Atividade Prática Aula Pandas Essencial**

**Atividade Prática Aula 2**

In [ ]:
# Atividade Prática — Aula 2: Pandas Essencial
**Dataset:** `vendas_dataviz_aula2.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuração para melhor visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

**2. Leitura do dataset**

In [ ]:
df = pd.read_csv('vendas_dataviz_aula2.csv')
df.head()

**3. Anatomia dos dados: DataFrame e Series**

In [ ]:
# Questão 1
print("1. O que é um DataFrame?")
print("→ É uma estrutura tabular bidimensional (como uma planilha do Excel) com linhas e colunas.")

print("\n2. O que é uma Series?")
print("→ É uma estrutura unidimensional (como uma coluna do Excel).")

# Exemplo prático
serie_produto = df['produto']
print("\nTipo da coluna 'produto':", type(serie_produto))
serie_produto.head()

**4. Check-up inicial do dataset**

In [ ]:
# Questão 2
print("Shape do dataset:", df.shape)
print(f"O dataset possui {df.shape[0]:,} linhas e {df.shape[1]} colunas.\n")

print("Informações gerais:")
df.info()

print("\nTipos de dados:")
print(df.dtypes)

Respostas:

Quantas linhas e colunas? → 300 linhas e 11 colunas
Colunas numéricas: vendas, quantidade, preco_unitario, custo, lucro, avaliacao_cliente
Problemas: várias colunas com tipo object (texto) que deveriam ser numéricas

**5. Inspeção de problemas de qualidade**

In [ ]:
# Questão 3
print("Valores nulos por coluna:")
print(df.isna().sum())

print("\n=== Canal ===")
print(df['canal'].value_counts(dropna=False))

print("\n=== Estado ===")
print(df['estado'].value_counts(dropna=False))

# Verificar valores não numéricos em 'lucro' e 'vendas'
print("\nValores suspeitos em 'lucro':")
print(df['lucro'].unique()[:20])  # mostra alguns valores

# Converter lucro para numérico e verificar infinitos
df['lucro_num'] = pd.to_numeric(df['lucro'], errors='coerce')
print("\nQuantidade de valores infinitos em lucro:", np.isinf(df['lucro_num']).sum())

Problemas identificados:

Variações no canal: Marketplace, MarketPlace, Online, online, ONLINE
Estados com letras minúsculas (rj)
Alguns valores em lucro e vendas com texto ("R$ 801,60", "inf")
Valores ausentes em algumas colunas

**6. Selecionando o que importa (df_dash)**

In [ ]:
# Questão 4
cols_dashboard = ['data', 'estado', 'canal', 'produto', 'categoria', 'vendas', 'lucro']
df_dash = df[cols_dashboard].copy()
df_dash.head()

**7. Preparação mínima para análise**

In [ ]:
# Questão 5
# Converter data
df_dash['data'] = pd.to_datetime(df_dash['data'], errors='coerce')
df_dash['mes'] = df_dash['data'].dt.to_period('M')

# Limpeza da coluna 'vendas'
vendas_limpa = (
    df_dash['vendas']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('.', '', regex=False)   # remove ponto de milhar
    .str.replace(',', '.', regex=False)  # troca vírgula decimal
    .str.strip()
)

df_dash['vendas_num'] = pd.to_numeric(vendas_limpa, errors='coerce')

# Converter lucro para numérico
df_dash['lucro_num'] = pd.to_numeric(df_dash['lucro'], errors='coerce')

df_dash.head()

**8. Filtragem simples: estado RJ**

In [ ]:
# Questão 6
df_rj = df_dash[df_dash['estado'] == 'RJ']
print("Registros no RJ:", df_rj.shape[0])
print("Receita total no RJ: R$", df_rj['vendas_num'].sum().round(2))

**9. Filtragem com múltiplas condições**

In [ ]:
# Questão 7
# Forma 1: operadores lógicos
df_rj_online = df_dash[(df_dash['estado'] == 'RJ') & (df_dash['canal'] == 'Online')]

# Forma 2: query()
df_rj_online_q = df_dash.query("estado == 'RJ' and canal == 'Online'")

print("Registros RJ + Online:", df_rj_online.shape[0])
df_rj_online.head()

**10. Priorização: rankings**

In [ ]:
# Questão 8
print("=== Top 10 maiores vendas ===")
top10_vendas = df_dash.nlargest(10, 'vendas_num')
display(top10_vendas)

print("\n=== Top 10 maiores lucros ===")
top10_lucro = df_dash.nlargest(10, 'lucro_num')
display(top10_lucro)

print("\n=== 5 menores lucros ===")
bottom5 = df_dash.nsmallest(5, 'lucro_num')
display(bottom5)

11. Ranking por produto

In [ ]:
# Questão 9
ranking_produtos = (
    df_dash.groupby('produto', dropna=False)
    .agg(
        vendas_total=('vendas_num', 'sum'),
        lucro_total=('lucro_num', 'sum'),
        quantidade_vendas=('vendas_num', 'count')
    )
    .round(2)
    .sort_values('lucro_total', ascending=False)
)

print("Top 10 produtos mais lucrativos:")
ranking_produtos.head(10)

**12. Ranking por estado e canal**

In [ ]:
# Questão 10
ranking_estados = (
    df_dash.groupby('estado', dropna=False)['vendas_num']
    .sum()
    .sort_values(ascending=False)
)

ranking_canais = (
    df_dash.groupby('canal', dropna=False)['lucro_num']
    .sum()
    .sort_values(ascending=False)
)

print("Ranking de Estados por Receita:")
print(ranking_estados)

print("\nRanking de Canais por Lucro Total:")
print(ranking_canais)

**13. Análise temporal simples**

In [ ]:
# Questão 11
receita_mensal = df_dash.groupby('mes')['vendas_num'].sum().sort_index()

print("Receita mensal:")
print(receita_mensal)

# Gráfico
receita_mensal.plot(kind='bar', figsize=(12, 5), color='skyblue')
plt.title('Receita Total por Mês (2024)')
plt.ylabel('Receita (R$)')
plt.xlabel('Mês')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

 **14. Interpretação Gerencial (célula de texto)
Interpretações:**

Produto mais lucrativo:
O produto Notebook Pro (da categoria Informática) é disparadamente o mais lucrativo. Ele gera muito mais lucro absoluto que os demais. Recomendação: priorizar estoque, campanhas e margem nesse produto.
Estado com maior receita:SP (São Paulo) lidera com folga a receita total. Isso era esperado, pois é o maior mercado consumidor do Brasil. Sugestão: reforçar presença e logística em SP.
Canal com maior lucro:Loja Física aparece como o canal mais lucrativo na maioria das análises, seguido por Online. Marketplace tem volume, mas margem menor.
Problema de qualidade encontrado:
Existem valores inconsistentes em canal (Marketplace × MarketPlace, online em diferentes grafias) e alguns valores textuais em colunas numéricas (R$ e "inf"). Isso pode quebrar dashboards se não for tratado.

**15. Desafio Extra (Limpeza Avançada)**

In [ ]:
# Desafio extra - Limpeza adicional
df_clean = df_dash.copy()

# Padronizar canal
df_clean['canal'] = df_clean['canal'].astype(str).str.strip().str.lower()
df_clean['canal'] = df_clean['canal'].replace({
    'marketplace': 'marketplace',
    'marketplace ': 'marketplace',
    'online': 'online',
    'online ': 'online'
})

# Padronizar estado (maiúsculo)
df_clean['estado'] = df_clean['estado'].astype(str).str.strip().str.upper()

# Tratar valores infinitos em lucro
df_clean['lucro_num'] = df_clean['lucro_num'].replace([np.inf, -np.inf], np.nan)

print("Após limpeza:")
print("Canais únicos:", df_clean['canal'].unique())
print("Estados únicos:", df_clean['estado'].unique())
print("Valores nulos em lucro:", df_clean['lucro_num'].isna().sum())

# Novo ranking após limpeza
ranking_canais_clean = df_clean.groupby('canal')['lucro_num'].sum().sort_values(ascending=False)
print("\nRanking de Canais após limpeza:")
print(ranking_canais_clean)